In [1]:
import json
import os
from datetime import datetime

from torch.utils.tensorboard import SummaryWriter

from msi_vae.models import *
from msi_vae.msi_dataset import *
from msi_vae.trainer import *
from msi_vae.data import *

In [2]:
path = os.path.join(
    os.environ["HOME"],
    ".datasets",
    "mass_spec_data",
    "Cancer biopsy",
    "5 June",
    "binned_with_tic.h5",
)

if not os.path.exists(path):
    raise FileNotFoundError(f"The preprocessed data is not present at {path}")

In [3]:
input_dim = 70000
latent_dim = 200
learning_rate = 1e-3
weight_decay = 1e-2
batch_size = 64
num_epochs = 20
beta = 1
dimensions = [2048, 512]
loss_type = "BCE"

hparams = {
    "input_dim": input_dim,
    "latent_dim": latent_dim,
    "learning_rate": learning_rate,
    "weight_decay": weight_decay,
    "batch_size": batch_size,
    "num_epochs": num_epochs,
    "beta": beta,
    "dimensions_1": dimensions[0],
    "dimensions_2": dimensions[1],
    "path": path,
    "loss_type": loss_type
}

In [4]:
DEVICE = get_torch_device()
DEVICE

device(type='mps')

In [5]:
train_data = MSIDataset(
    path,
    "train",
    input_dim,
)

train_loader = torch.utils.data.DataLoader(
    train_data,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0,
)

test_loader = torch.utils.data.DataLoader(
    train_data,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0,
)

assert len(train_loader) == len(test_loader)
print(f"Number of spectra: {len(train_data)}")
print(f"Number of batches: {len(train_loader)}")

Number of spectra: 160000
Number of batches: 2500


In [6]:
for batch in train_loader:
    print("Batch shape:", batch.shape)
    break

Batch shape: torch.Size([64, 70000])


In [7]:
model = VAE(input_dim, latent_dim, beta, dimensions, loss_type).to(DEVICE)
model

VAE(
  (linear_1_enc): Linear(in_features=70000, out_features=2048, bias=True)
  (linear_1_enc_batch_norm): BatchNorm1d(2048, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (linear_2_enc): Linear(in_features=2048, out_features=512, bias=True)
  (linear_2_enc_batch_norm): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (mean): Linear(in_features=512, out_features=200, bias=True)
  (mean_batch_norm): BatchNorm1d(200, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (log_var): Linear(in_features=512, out_features=200, bias=True)
  (log_var_batch_norm): BatchNorm1d(200, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (linear_1_dec): Linear(in_features=200, out_features=512, bias=True)
  (linear_1_dec_batch_norm): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (linear_2_dec): Linear(in_features=512, out_features=2048, bias=True)
  (linear_2_dec_batch_norm): BatchNorm1

In [8]:
optimizer = torch.optim.AdamW(
    model.parameters(), lr=learning_rate, weight_decay=weight_decay
)

In [9]:
def run_trainer(run_id, assets_path):
    results_path = os.path.join(assets_path, "weights")
    writer = SummaryWriter(os.path.join(assets_path, "tensorboard", run_id))

    writer.add_text("hyperparameters", json.dumps(hparams, indent=4))

    batches_processed = 0
    samples_processed = 0
    best_loss = torch.inf

    for epoch in range(num_epochs):
        batches_processed, samples_processed, average_epoch_loss = train(
            model,
            train_loader,
            optimizer,
            writer,
            samples_processed,
            batches_processed,
            5,
            DEVICE,
        )
        print(
            f"\n====> Epoch: {epoch+1}/{num_epochs}, Average loss: {average_epoch_loss:.12f}\n"
        )

        if average_epoch_loss < best_loss:
            model_path = os.path.join(results_path, f"{run_id}.pth")
            print(f"Saving the model to {model_path}\n")
            os.makedirs(results_path, exist_ok=True)
            torch.save(model.state_dict(), model_path)
            best_loss = average_epoch_loss

    writer.close()

In [10]:
moment_run = datetime.now().strftime("%Y%m%d-%H%M%S")
run_id = f"vae_msi_{moment_run}"

assets_path = os.path.join(os.path.dirname(os.getcwd()), "assets")
os.makedirs(assets_path, exist_ok=True)

save_hyperparameters(hparams, assets_path, run_id)

run_trainer(run_id, assets_path)

  0%|          | 0/2500 [00:00<?, ?it/s]

Step: 0 (Samples: 0), Loss: 3171827.000000000000 (Recon: 3160565.500000000000, KL: 11261.488281250000)
Step: 500 (Samples: 32000), Loss: 8380.113281250000 (Recon: 595.792602539062, KL: 7784.320312500000)
Step: 1000 (Samples: 64000), Loss: 4226.196777343750 (Recon: 535.241882324219, KL: 3690.954833984375)
Step: 1500 (Samples: 96000), Loss: 2255.457275390625 (Recon: 519.611877441406, KL: 1735.845336914062)
Step: 2000 (Samples: 128000), Loss: 1395.618408203125 (Recon: 505.310882568359, KL: 890.307556152344)
Step: 2499 (Samples: 159936), Loss: 1079.156860351562 (Recon: 502.772827148438, KL: 576.384033203125)

====> Epoch: 1/20, Average loss: 146.160949707031

Saving the model to /Users/jmanwillz/Desktop/temp/RMCP-Project/assets/weights/vae_msi_20251218-181743.pth



  0%|          | 0/2500 [00:00<?, ?it/s]

Step: 0 (Samples: 160000), Loss: 1066.595703125000 (Recon: 490.555419921875, KL: 576.040222167969)
Step: 500 (Samples: 192000), Loss: 968.322448730469 (Recon: 501.564270019531, KL: 466.758178710938)
Step: 1000 (Samples: 224000), Loss: 900.436401367188 (Recon: 493.587585449219, KL: 406.848846435547)
Step: 1500 (Samples: 256000), Loss: 867.594116210938 (Recon: 495.232238769531, KL: 372.361846923828)
Step: 2000 (Samples: 288000), Loss: 835.910400390625 (Recon: 496.012756347656, KL: 339.897613525391)
Step: 2499 (Samples: 319936), Loss: 808.825561523438 (Recon: 497.919067382812, KL: 310.906524658203)

====> Epoch: 2/20, Average loss: 14.032475471497

Saving the model to /Users/jmanwillz/Desktop/temp/RMCP-Project/assets/weights/vae_msi_20251218-181743.pth



  0%|          | 0/2500 [00:00<?, ?it/s]

...

Step: 0 (Samples: 2720000), Loss: 487.983581542969 (Recon: 487.981323242188, KL: 0.002243936062)
Step: 500 (Samples: 2752000), Loss: 487.972290039062 (Recon: 487.970764160156, KL: 0.001525521278)
Step: 1000 (Samples: 2784000), Loss: 492.254272460938 (Recon: 492.251770019531, KL: 0.002507060766)
Step: 1500 (Samples: 2816000), Loss: 512.718322753906 (Recon: 512.716064453125, KL: 0.002252697945)
Step: 2000 (Samples: 2848000), Loss: 500.464294433594 (Recon: 500.463134765625, KL: 0.001174122095)
Step: 2499 (Samples: 2879936), Loss: 495.833801269531 (Recon: 495.831390380859, KL: 0.002415210009)

====> Epoch: 18/20, Average loss: 7.632343769073

Saving the model to /Users/jmanwillz/Desktop/temp/RMCP-Project/assets/weights/vae_msi_20251218-181743.pth



  0%|          | 0/2500 [00:00<?, ?it/s]

Step: 0 (Samples: 2880000), Loss: 489.029113769531 (Recon: 489.026885986328, KL: 0.002233594656)
Step: 500 (Samples: 2912000), Loss: 482.444885253906 (Recon: 482.439636230469, KL: 0.005248606205)
Step: 1000 (Samples: 2944000), Loss: 487.134277343750 (Recon: 487.132904052734, KL: 0.001373022795)
Step: 1500 (Samples: 2976000), Loss: 477.506713867188 (Recon: 477.505126953125, KL: 0.001588016748)
Step: 2000 (Samples: 3008000), Loss: 473.827453613281 (Recon: 473.825866699219, KL: 0.001572668552)
Step: 2499 (Samples: 3039936), Loss: 493.801422119141 (Recon: 493.798950195312, KL: 0.002471238375)

====> Epoch: 19/20, Average loss: 7.631758689880

Saving the model to /Users/jmanwillz/Desktop/temp/RMCP-Project/assets/weights/vae_msi_20251218-181743.pth



  0%|          | 0/2500 [00:00<?, ?it/s]

Step: 0 (Samples: 3040000), Loss: 499.833801269531 (Recon: 499.830932617188, KL: 0.002875566483)
Step: 500 (Samples: 3072000), Loss: 496.748748779297 (Recon: 496.746124267578, KL: 0.002621769905)
Step: 1000 (Samples: 3104000), Loss: 485.248199462891 (Recon: 485.246765136719, KL: 0.001423120499)
Step: 1500 (Samples: 3136000), Loss: 493.172821044922 (Recon: 493.169525146484, KL: 0.003283113241)
Step: 2000 (Samples: 3168000), Loss: 505.794860839844 (Recon: 505.792327880859, KL: 0.002530962229)
Step: 2499 (Samples: 3199936), Loss: 486.622619628906 (Recon: 486.619598388672, KL: 0.003012955189)

====> Epoch: 20/20, Average loss: 7.631203174591

Saving the model to /Users/jmanwillz/Desktop/temp/RMCP-Project/assets/weights/vae_msi_20251218-181743.pth

